# Hong Kong Ghost Signs

## Imports, Config, and Constants

In [153]:
import os
import json
from random import choice
from datetime import date

# Datetime manipulation
from datetime import datetime
import datefinder

# Dataframe manipulation
import polars as pl
from polars.exceptions import ComputeError
from polars.datatypes import Datetime, String

# Google Maps and Translate API Clients
import googlemaps
from googletrans import Translator
from google.cloud import translate as google_translate

# Generate UUID
import shortuuid

# Distance Calculation
import haversine as hs
from haversine import Unit


# Setup Google Translator
translator = Translator()

# Configure Polars 
cfg = pl.Config()
cfg.set_tbl_rows(32)
cfg.set_fmt_str_lengths(256)
cfg.set_tbl_width_chars(1000)

# Define Hong Kong Bounding Box
HKBBOX = (22.5639, 114.4013, 22.1771, 113.8373) 

# PATHS to CSVs
DATASRC = "data/hkghostmaps-gmaps-export-20220901.csv"
DATASRC_UUID = "data/hkghostmaps-gmaps-prepared-20220905+uuid.csv"
DATADEST = f"data/export/hkghostmaps-gmaps-prepared-{datetime.isoformat(datetime.now())[:10]}.csv"
JSONDEST = f"data/export/hkghostmaps-gmaps-prepared-{datetime.isoformat(datetime.now())[:10]}.json"

## Load Data
### Generate UUID on first load

In [154]:
if not os.path.isfile(DATASRC_UUID):
    # Load Features
    df = pl.read_csv(
        source=DATASRC,
        # Prefix columns with 'raw'
        new_columns=[
            'rawX',
            'rawY',
            'rawName',
            'rawDesc',
            'rawGxml'
        ]).filter(
            # Accept Missing Values
            pl.col("rawX").eq(0) | (
            # Filter Non-Hong Kong Markers
            (HKBBOX[3] <= pl.col("rawX")) &
            (pl.col("rawX") <= HKBBOX[1]) &
            (HKBBOX[2] <= pl.col("rawY")) &
            (pl.col("rawY") <= HKBBOX[0]))
        ).sort(by=('rawX','rawY'))

    # Add UUID
    def gen_shortuuid(v):
        return shortuuid.uuid()
    
    df = df.with_columns(
        id = pl.col('rawX').map_elements(gen_shortuuid, return_dtype=str)
    )

    df.write_csv(DATASRC_UUID)
else:
    df= pl.read_csv(
        DATASRC_UUID,
        new_columns=[
            'rawX',
            'rawY',
            'rawName',
            'rawDesc',
            'rawGxml',
            'id'
        ]
    )

# Text Parsing

## Parse "rawName"
### Identify the type of data and language in 'name' field
- Coordinates
- Address
- All Chinese
- Mixed Chinese & English
- All English

In [155]:
df = df.with_columns(
    rxNameIsCoordinates=pl.col('rawName').str.contains(r'^(\(22|22)'),
    rxNameIsAddress=pl.col('rawName').str.contains(r' Rd$| St$| Rd W$| Rd E$| Road$| Street$| Strand W$|Rd - Yuen Long$| Ln$'),
    rxNameIsAllChinese=(
        pl.col('rawName').str.contains(r'[\u4e00-\u9fff]+')
    ) & ~(pl.col('rawName').str.contains(r'[a-zA-Z]+')),
    rxNameIsChineseAndEnglish=(
        pl.col('rawName').str.contains(r'[\u4e00-\u9fff]+')
    ) & (pl.col('rawName').str.contains(r'[a-zA-Z]+'))
)

In [156]:
# In all other cases, treat the 'name' field to be English
df = df.with_columns(
    rxNameIsAllEnglish=(
        ~(df['rxNameIsCoordinates']) &
        ~(df['rxNameIsAddress']) &
        ~(df['rxNameIsAllChinese']) &
        ~(df['rxNameIsChineseAndEnglish'])
    )
)

## Set English Names from `EN` and mixed `EN/ZH-HANT` 'rawName' 

#### Set English from pure English

In [157]:
df = df.with_columns(
    pl.when(pl.col("rxNameIsAllEnglish"))
    .then(pl.col('rawName').str.to_titlecase().str.strip_chars())
    .otherwise(None).alias('name'),
    pl.when(pl.col("rxNameIsAllEnglish"))
    .then(False)
    .otherwise(None).alias('nameGen'),
)

#### Set English from Mixed English and Chinese

In [158]:
original = df.filter(df['rxNameIsChineseAndEnglish'])['rawName'].to_list()
translations = ["'Ltd. Co.' Under Newer 'Gospel Nursing Home' Sign",
"Illegible Blue Sign on Tile 'Center'",
"Prince 'Prince Edward' Jewelery",
"Sign Behind Air Conditioners Above 'Tai Hung Roast Goose'",
"'Chengdelong' Rice Shop",
"'Zhonghuayang' Illegible",
"HK.Fruit 'Global High-quality Ingredients (Fruits, Seafood, Hairy Crabs)",
"'College' Faded on the 6th Floor.",
"Side of 'Zuiqionglou' Hotel",
"Namkok 'South Point' (Cafe)"]

mixed_to_en = dict(zip(original,translations))

df = df.with_columns(
    pl.when(
        df['rxNameIsChineseAndEnglish']
    ).then(
        pl.col('rawName').replace_strict(mixed_to_en, default=None)
    ).otherwise(
        pl.col('name')
    ).alias('name'),
    # ~Human translated
    pl.when(
        df['rxNameIsChineseAndEnglish']
    ).then(
        False
    ).otherwise(
        pl.col('nameGen')
    ).alias('nameGen'),
)

In [159]:
# Result, with unprocessed rows
# df.select('name','nameGen').head(10)   

In [160]:
# Result, only processed rows
# df.filter(df['nameGen'] == False).select('name').head(10)

## Set Traditional Chinese Names from `ZH-HANT` 'rawName'

In [161]:
df = df.with_columns(
    pl.when(
        pl.col("rxNameIsAllChinese")
    ).then(
        pl.col('rawName').str.strip_chars()
    ).otherwise(
        None
    ).alias('name__zh-hant'),
    pl.when(
        pl.col("rxNameIsAllChinese")
    ).then(
        False
    ).otherwise(
        None
    ).alias('nameGen__zh-hant'),
)

In [162]:
# Result, only processed rows
# df.filter(df['nameGen__zh-hant'] == False).select('name__zh-hant').head(10)

## Translate Traditional Chinese Names to English

### Google Translate

In [163]:
original = [x for x in df['name__zh-hant'].to_list() if x]
translations = [f"'{translator.translate(x, src='zh-tw', dest='en').text}'" for x in original] 

In [164]:
zhhant_to_en = dict(zip(original,translations))

for src, dest in zhhant_to_en.items():
    print(f'{src} --> {dest}')

榮發行 --> 'Rong distribution'
元祥樓 --> 'Yuanxiang Tower'
水華實業有限公司 --> 'Shuihua Industrial Co., Ltd.'
棠烔盧 --> 'Tang Yulu'
八鄉鄉事會 --> 'Baxiang Township Club'
潮州同鄉會,天金同料行 --> 'Chaozhou Tongxiang Association, Tianjin Same as the same as'
鮮魚行商會， 李子然, 盧家氏會 --> 'Fresh Fish Chamber of Commerce, Li Ziran, Lu Jia's Association'
趙學...術會 --> 'Zhao Xue ... Submare'
永堂 --> 'Forever'
萬寶藥行 --> 'Montblanc'
元朗屯門報敗同業會有限 --> 'Yuanglang Tuen Mun Failure Failure Fair Co., Ltd.'
...器工程 --> 'Instrumental engineering'
興發茶餐廳（太平洋酒吧） --> 'Xingfa Tea Restaurant (Pacific Bar)'
福華茶行 --> 'Fuhua Tea'
堅強五金 --> 'Strong hardware'
除記 --> 'Remove'
松記大新時裝 --> 'Songji Daxin Fashion'
大信寶業 --> 'Daxinbaoye'
文記裝修工程 --> 'Wen Ji Decoration Project'
明華粉麵家 --> 'Minghua Fan Noodle Home'
鴻昌茶莊 --> 'Hongchang Tea Village'
英華百貨 --> 'Yinghua Department Store'
合記五金 --> 'Hardware'
生力 --> 'Vitality'
明駿冷氣工程公司 --> 'Mingjun Innonal Air Engineering Company'
興陸士多 --> 'Xinglu Shigo'
潘記木器油漆 --> 'Panji Wood'
嘉碧疋頭公司 --> 'Garbin Head Company'
大實力貨車及巴士 -->

In [165]:
df = df.with_columns(
    pl.when(
        df['rxNameIsAllChinese']
    ).then(
        pl.col('rawName').replace_strict(zhhant_to_en, default=None)
    ).otherwise(
        pl.col('name')
    ).alias('name'),
    # Machine translated
    pl.when(
        df['rxNameIsAllChinese']
    ).then(
        True
    ).otherwise(
        pl.col('nameGen')
    ).alias('nameGen'),
)

In [166]:
# Result, with unprocessed rows
# df.select('name','nameGen','rawName').head(10)   

## Parse "rawDesc"
### Identify the type of data and language in 'description' field
- Names (where missing `name`)
- Descriptions (where `name` exists too)
- Capture Date of the photo
- Image URL
- All Chinese
- Mixed Chinese & English
- All English

#### `imageUrl` and `visitableAsOf`

In [167]:
def extract_date(s):
    try:
        res = list(datefinder.find_dates(s, base_date=datetime(2023,1,1)))[0]
        return res
    except TypeError:
        return None
    except ComputeError:
        return None
    except IndexError:
        return None

pattern_img_elem = r"<img\b[^>]*>"
pattern_desc_img_url=r'src\s*=\s*"([^"]+)"'
pattern_desc_date=r'Captured'
pattern_desc_stripping_img_url_and_date = r"<br><br>(.*?)\s*Captured"
pattern_labeled_date=r"(Captured|Documented|Spotted)\s+\w+\s+\w*\s*(2022|2023|2024)\.?\W?"


# Parse raw name to identify the data entered in the 'description' field 
df = df.with_columns(
    # Image URLs
    rxImgUrlExtracted=pl.col('rawDesc').str.extract(pattern_desc_img_url),
    rxDescIsImgHtml=pl.col('rawDesc').str.contains(pattern_desc_img_url),
    # Capture date
    rxDescIscapturedDate=pl.col('rawDesc').str.contains(pattern_desc_date),
    rxvisitableAsOfExtracted=pl.col('rawDesc').map_elements(extract_date, return_dtype=Datetime),
    # Copy description for multi-step processing
    # ryDesc_before=pl.col('rawDesc').str.replace(pattern_img_elem,"").str.replace_all("<br>",""),
    ryDesc=pl.col('rawDesc').str.replace(pattern_img_elem,"").str.replace_all("<br>","").str.replace(pattern_labeled_date,"").str.strip_chars().replace("", None),
)

In [168]:
# Result, only Image URL
# df.filter(pl.col('rxDescIsImgHtml'))['rxImgUrlExtracted', 'rawDesc']

# Result, only Mixes English / Chinese
# df.filter(pl.col('rxDescIscapturedDate'))['rxvisitableAsOfExtracted','rawDesc']

In [169]:
# BEFORE and AFTER inspection
# pl.concat(
#     (df['ryDesc_before'].alias('BEFORE').to_frame(),
#      df['ryDesc'].str.replace(pattern_labeled_date,"").alias('AFTER').to_frame()
#     ),how='horizontal')

In [170]:
# Manual
manual_desc_encoding = {
    "Corner of Wing Fung & Queen's Road East. Captured while hoarding was up for new development in late April 2023." : "Captured while hoarding was up for new development in late April 2023."
}

df = df.with_columns(
    pl.when(
        pl.col('ryDesc').is_in(list(manual_desc_encoding.keys()))
    ).then(
        pl.col('ryDesc').replace_strict(manual_desc_encoding, default=None)       
    ).otherwise(
        pl.col('ryDesc')
    )
)

#### Classsify Languages

In [171]:
df = df.with_columns(
    ryDescIsAllChine=(
        pl.col('ryDesc').str.contains(r'[\u4e00-\u9fff]+')
    ) & ~(pl.col('ryDesc').str.contains(r'[a-zA-Z]+')),
    ryDescIsChineseAndEnglish=(
        pl.col('ryDesc').str.contains(r'[\u4e00-\u9fff]+')
    ) & (pl.col('ryDesc').str.contains(r'[a-zA-Z]+')
        )
)

# In all other cases, treat the 'description' field to be English
df = df.with_columns(
    ryDescIsAllEnglish=(
        df['ryDesc'].is_not_null() &
        ~(df['ryDescIsAllChine']) &
        ~(df['ryDescIsChineseAndEnglish'])
    )
)

In [172]:
# Result, All Chinese
# df.filter(pl.col('ryDescIsAllChine'))['ryDesc']
# Result, Mixed English / Chinese
# df.filter(pl.col('ryDescIsChineseAndEnglish'))['ryDesc']
# Result, All English
# df.filter(pl.col('ryDescIsAllEnglish'))['ryDesc']

## Determine which All English descriptions can be promoted to the English name

In [173]:
desc_to_name_exceptions = [
    "Revealed sign after a car accident broke off a chunk of the building.",
    "3 Ghost Signs - trading company, plumbing and electrician, garment factory.",
    "Dragon Phoenix Photography Studio and unreadable sign remnant on concrete."
]

desc_to_name_exception_names = {
    "Revealed sign after a car accident broke off a chunk of the building." : "Revealed By Car Accident",
    "3 Ghost Signs - trading company, plumbing and electrician, garment factory." : "Ghost Sign Trio",
    "Dragon Phoenix Photography Studio and unreadable sign remnant on concrete." : "Dragon Phoenix Photography Studio",
    "'同' above Healthy Wood on the side next to garage" : "Together"
}

desc_to_name_exception_descs = {
    "3 Ghost Signs - trading company, plumbing and electrician, garment factory." : "Trading Company, Plumber and Electrician, Garment Factory",
    "Dragon Phoenix Photography Studio and unreadable sign remnant on concrete." : "Along with an unreadable sign remnant on concrete",
    "(one day before it was scheduled to be painted over)" : "The day before it was scheduled to be painted over",
    "寶 something trading - vacant shop with for lease sign over the shop front" : "Vacant shop with for lease sign over the shop front",
    "金堂大廈 sign under air conditioning units over 3 shop" : "Under air conditioning units over 3 shop",
    '億和公司 revealed after vegetable stall closed'  : "Revealed after vegetable stall closed",
    '地產代理 in white in side alley.': "In white, found in a side alley",
    '灸 next to Stern Rockwell.' : "Next to Stern Rockwell",
    'Duplicate jeans sign of the one being covered down the road Po Sang Trading 寶生易公司' : "Duplicate jeans sign of the one being covered down the road Po Sang Trading",
    '中國肉公司 - behind roll up gate.' : "Behind a roll up gate",
    '陳董仁 painted over in yellow on Reclamation Street' : "Painted over in yellow on Reclamation Street",
    "'智' on corner of Nanking and Shanghai Street" : "On corner of Nanking and Shanghai Street",
    "Chinese Medicine and '牛王' red letters on blue tile" : "Red letters on blue tile",
    "'同' above Healthy Wood on the side next to garage" : "Above Healthy Wood on the side next to garage"
}

df = df.with_columns(
    pl.when(
        (pl.col('name').is_null()) &
        (pl.col('ryDescIsAllEnglish')) &
        (pl.col('name').is_null()) &
        ~(pl.col('ryDesc').is_in(desc_to_name_exceptions))
    ).then(
        pl.col('ryDesc').str.to_titlecase().str.strip_chars().str.replace(r'$\(','').str.replace(r'\)^','')
    ).otherwise(
        pl.col('name')
    ).alias(
        'name'
    ),
    # Set Name Generated Field to False
    pl.when(
        (pl.col('name').is_null()) &
        (pl.col('ryDescIsAllEnglish')) &
        (pl.col('name').is_null()) &
        ~(pl.col('ryDesc').is_in(desc_to_name_exceptions))
    ).then(
        False
    ).otherwise(
        pl.col("nameGen")
    ).alias('nameGen'),
    # Set Description Field to None -- silly code
    pl.when(
        True      
    ).then(
        None
    ).otherwise(
        None
    ).alias('description'),   
    # Set Description Generated Field to False
    pl.when(
        True
    ).then(
        None
    ).otherwise(
        None
    ).alias('descriptionGen')   
)

# Manual Correction of Names
df = df.with_columns(
    pl.when(
        pl.col('ryDesc').is_in(list(desc_to_name_exception_names.keys()))
    ).then(
        pl.col('ryDesc').replace_strict(desc_to_name_exception_names, default=None)  
    ).otherwise(
        pl.col('name')
    ).alias('name'),
    pl.when(
        pl.col('ryDesc').is_in(list(desc_to_name_exception_names.keys()))
    ).then(
        False
    ).otherwise(
        pl.col('nameGen')
    ).alias('nameGen'),
)

# Manual Correction of Descriptions
df = df.with_columns(
    pl.when(
        pl.col('ryDesc').is_in(list(desc_to_name_exception_descs.keys()))
    ).then(
        pl.col('ryDesc').replace_strict(desc_to_name_exception_descs, default=None)    
    ).otherwise(
        pl.col('description')
    ).alias('description'),
    pl.when(
        pl.col('ryDesc').is_in(list(desc_to_name_exception_descs.keys()))
    ).then(
        False
    ).otherwise(
        pl.col('descriptionGen')
    ).alias('descriptionGen'),
)


## Determine which Mixed English / Chinese descriptions can be promoted to the English name

In [174]:
original_still_missing_name = df.filter((df['name'].is_null()) & (df['ryDescIsChineseAndEnglish']))['ryDesc'].to_list()

translations = [
    "'Precious' Something Trading",
    "'Jintang Building'",
    "'Yihe Company'",
    "'Estate agency'",
    "'Moxibustion'",
    "Behind 'Cheng Hing' and 'Defa Tea Restaurant'",
    "Multiple 'Runfa' Signs",
    "'Jeans'",
    "Illegible 'Company'",
    "Side of building 'Country _ Fixed Head'",
    "'Big Welding Supply Co.'",
    "'Yajiu Engineering Co. Ltd.'",
    "'China Meat Company'",
    "'Chen Dongren'",
    "'Wisdom'",
     "'West Branch', then Illegible",
    "Chinese Medicine and 'Ox King'",
   "'Together'",
 ]

mixed_to_en = dict(zip(original_still_missing_name,translations))

df = df.with_columns(
    pl.when(
       (df['name'].is_null()) & (df['ryDescIsChineseAndEnglish'])
    ).then(
        pl.col('ryDesc').replace_strict(mixed_to_en, default=None)
    ).otherwise(
        pl.col('name')
    ).alias('name'),
    # ~Human translated
    pl.when(
       (df['name'].is_null()) & (df['ryDescIsChineseAndEnglish'])
    ).then(
        False
    ).otherwise(
        pl.col('nameGen')
    ).alias('nameGen'),
)

In [175]:
# Results, processed rows
# df.filter((df['name'].is_not_null()) & (df['ryDescIsChineseAndEnglish']))['name','description', 'ryDesc']

## Determine which Mixed English / Chinese descriptions can be translated and used as the English description

In [176]:
original_already_have_names = df.filter(
        (df['nameGen'] != False) & 
        (df['name'].is_not_null()) & 
        (df['ryDescIsChineseAndEnglish'])
    )['ryDesc'].to_list()

translations = [
 "Above dessert place",
 "In the back alley behind Ping Pong",
 "In yellow",
 "Illegible 'hardware' - behind car repair shop sign",
 "Double layer with paint peeling off to reveal 'Lu Apartment Shanghai'",
 "'Lin_Wen` Chinese Doctor and 2 or 3 more illegible to the right",
 "'Irobu' painted over sign - off Jordan Road",
 "'Peace' under signage from old Standard Chartered Branch"
]

mixed_to_en = dict(zip(original_already_have_names,translations))

df = df.with_columns(
    pl.when(
        (df['nameGen'] != False) & (df['name'].is_not_null()) & (df['ryDescIsChineseAndEnglish'])
    ).then(
        pl.col('ryDesc').replace_strict(mixed_to_en, default=None)
    ).otherwise(
        pl.col('description')
    ).alias('description'),
    # ~Human translated
    pl.when(
       (df['nameGen'] != False) & (df['name'].is_not_null()) & (df['ryDescIsChineseAndEnglish'])
    ).then(
        False
    ).otherwise(
        pl.col('descriptionGen')
    ).alias('descriptionGen'),
)

In [177]:
# Results, processed rows
# df.filter((df['description'].is_not_null()) & (df['ryDescIsChineseAndEnglish']))['id','name','description','ryDesc']

In [178]:
# Manual correction / cleanup
name_for_id = {
    "JSrHippq8sn6487Soy57Uj" : "'Fuhua Tea Shop'",
    "mWFQstFZ7EHYZoa4UoF4vb" : "'Ming Hua Noodle House'",
    "PrhRzZGd3Q3WPUqTZzAaPD" : "'He Kee Hardware'",
    "GRARP8Q9YhUMZpybV4bHwX" : "'Pan Ji Wood Paint'"
}

df = df.with_columns(
    pl.when(
        df['id'].is_in(list(name_for_id.keys()))
    ).then(
        pl.col('id').replace_strict(name_for_id, default=None)
    ).otherwise(
        pl.col('name')
    ).alias('name')
)

## Set Traditional Chinese names from `ZH-HANT` 'rawDesc'

In [179]:
df = df.with_columns(
    pl.when(
        (pl.col("ryDescIsAllChine")) &
        (pl.col('name__zh-hant').is_null())
    ).then(
        pl.col('ryDesc').str.strip_chars().str.strip_chars('.')
    ).otherwise(
        pl.col('name__zh-hant')
    ).alias('name__zh-hant'),
    
    pl.when(
        (pl.col("ryDescIsAllChine")) &
        (pl.col('name__zh-hant').is_null())
    ).then(
        False
    ).otherwise(
        pl.col('nameGen__zh-hant')
    ).alias('nameGen__zh-hant')
)

In [180]:
# Result, only processed rows
# df.filter((pl.col("ryDescIsAllChine")) & (df['nameGen__zh-hant'] == False)).select('name__zh-hant')

## Determine which Mixed English / Chinese descriptions can be used as the Traditional Chinese name

In [181]:
mapping_id_to_name_zh_hant = {
    "MoUenBfFNz8kwiC9cYv7xE" : "'寶__'",
    "SXzbKw562zzZMGxrMeD2b2" : "'金堂大廈'",
    "h6H8bHYcNCMRNu6JQZtZas" : "'億和公司'",
    "nzAhQWc66HPtsif9nfffeU" : "'成德隆'",
    "b5xgzdeidUsg7edwKBCBx9" : "'地產代理'",
    "5CGpNw9S78bvcXHCB5TyXv" : "'灸'",
    "aw7wLFXS6gYrP9sv63Sfok" : "'成興和德發茶餐廳'",
    "mCEPQKUwFCy8cYZkjuRbBf" : "'潤發行'",
    "jueyvJ3o9mVo7QbQpEDFCV" : "'寶生易公司'",
    "J3ZgttCaPZV8STErD5Rec4" : "難以辨認'公司'",
    "fFLpYi6t9TL3VG62JRzaDg" : "'國_定頭'",
    "U7PDZpzB9HCYAaP2tM6BAw" : "'中華洋'難以辨認",
    "TjSqbaRW8REbwjiQWkqyR7" : "'大溶接器材行'",
    "bA3Xh4sivm59KjQMow5j8w" : "'雅就工程有限公司'",
    "7dFfj8vVzee6ZaTHBLCEVg" : "'陳董仁'",
    "4QEUmAmDHbXxohLBbewewT" : "'智'",
    "gmyBUNrK6gARB9qP5gm3E8" : "'西可'",
    "AXPkkwLhy7SoMYAFEs7KLu" : "'牛王'",
    "oGoev2mBpKKCu82CYNqGKQ" : "'學院'",
    "JuxA3HQde3Gi4WdrsEyEsN" : "'醉瓊楼飯店'",
    "axrHTxU4wm8KHPCCKFdu9F" : "'尖尖照相'",
    "h2tw4jFNCFLMZENhV6btKf" : "'龍'不完整的",
    "Eu5gBHGa7c4AWo4f7fp7Lg" : "'同'"
}

df = df.with_columns(
    pl.when(
        df['id'].is_in(list(mapping_id_to_name_zh_hant.keys()))
    ).then(
        pl.col('id').replace_strict(mapping_id_to_name_zh_hant, default=None)
    ).otherwise(
        pl.col('name__zh-hant')
    ).alias('name__zh-hant'),
    
    pl.when(
        df['id'].is_in(list(mapping_id_to_name_zh_hant.keys()))
    ).then(
        False
    ).otherwise(
        pl.col('nameGen__zh-hant')
    ).alias('nameGen__zh-hant')
)

In [182]:
# Result, processed rows
# df.filter(
#    (df['name__zh-hant'].is_not_null()) &  
#    (df['ryDescIsChineseAndEnglish'])
# )['id','name__zh-hant','nameGen__zh-hant','ryDesc']

## Promote Traditional Chinese descriptions from `rawDesc` to description, if they differ from the Traditional Chinese Name

In [183]:
df = df.with_columns(
    pl.when(
        pl.col('rxNameIsAllChinese') &
        pl.col('ryDescIsAllChine') &
        ((pl.col('name__zh-hant')) != (pl.col('ryDesc').str.strip_chars('.')))
    ).then(
        pl.col('ryDesc')
    ).otherwise(
        None
    ).alias('description__zh-hant'),
    # NOT Machine translated
    pl.when(
        pl.col('rxNameIsAllChinese') &
        pl.col('ryDescIsAllChine') &
        ((pl.col('name__zh-hant')) != (pl.col('ryDesc').str.strip_chars('.')))
    ).then(
        False
    ).otherwise(
        None
    ).alias('descriptionGen__zh-hant'),
)

In [184]:
# Result, 
# df[
#  'name',
#  'nameGen',
#  'name__zh-hant',
#  'nameGen__zh-hant',
#  'description',
#  'descriptionGen',
# 'description__zh-hant',
# 'descriptionGen__zh-hant',
#  'rawName',
#  'ryDesc',
# 'rxNameIsAllChinese',
#  'ryDescIsAllChine'
# ].filter(
#     pl.col('rxNameIsAllChinese') &
#     pl.col('ryDescIsAllChine') &
#     ((pl.col('name__zh-hant')) != (pl.col('ryDesc').str.strip_chars('.')))
# )


## Translate English names from `ZH-HANT` 'rawDesc' 

### Google Translate

In [185]:
original_still_missing_name = df.filter(
    (df['name'].is_null()) & 
    (df['ryDescIsAllChine'])
)['ryDesc'].to_list()

translations = [f"'{translator.translate(x, src='zh-tw', dest='en').text}'" for x in original_still_missing_name]

zhhant_to_en = dict(zip(original_still_missing_name,translations))

for src, dest in zhhant_to_en.items():
    print(f'{src} --> {dest}')

業蛇 --> 'Karma'
老福德宮聯誼會. --> 'Old Ford Palace Association.'
名美茶莊 --> 'Famous tea house'
秦華行 --> 'Qin Huaxing'
新東榮小菜王 --> 'Xindong Rongmi Kings'
堂生 --> 'Dignified'
洪合樟木槓廠 --> 'Honghe Camphor Wood Bar Factory'
順 --> 'Go'
中發 --> 'Medium'
義達有業公司 --> 'Yida You Industrial Company'
錦光電器批_ --> 'Jinuang Electric Approval_'
安利浩記五金 --> 'Amway Haoji Hardware'
大成三樓 --> 'Dacheng third floor'
田稠趙群洽堂 --> 'Tianchong Zhao Qunqiang'
花悅. --> 'Huayue.'
志成地產 --> 'Zhicheng Real Estate'
馮記電器 --> 'Feng Ji Electric'
文鴻 --> 'Wenhong'
龍鳳➡️ --> 'Dragon and Phoenix ➡️'
文官 --> 'Civilian'
新勝昌 --> 'Xin Shengchang'
鴻湍記 --> 'Hong Tuanji'
金花攝影. --> 'Golden Flower Photography.'
黎醒民 --> 'Li Xingmin'
李大恩中醫 --> 'Li Daen Traditional Chinese Medicine'
泰蔣寶館 --> 'Thailand'
正道 --> 'right way'
大 --> 'big'
'中' --> ''middle''
東尼髮屋. --> 'Tony Fay House.'


In [186]:
df = df.with_columns(
    pl.when(
        (df['ryDescIsAllChine'])&
        (df['name']).is_null()
    ).then(
        pl.col('ryDesc').replace_strict(zhhant_to_en, default=None).str.to_titlecase()
    ).otherwise(
        pl.col('name')
    ).alias('name'),
    # Machine translated
    pl.when(
        df['ryDescIsAllChine']
    ).then(
        True
    ).otherwise(
        pl.col('nameGen')
    ).alias('nameGen'),
)

In [187]:
# Result, only processed rows
# df.filter(
#     (df['ryDescIsAllChine']) &
#     (df['nameGen'])
# ).select('name','nameGen','ryDesc')

## Translate English descriptions from `ZH-HANT` 'rawDesc' 

In [188]:
original_already_have_names_but_missing_desc = df.filter(
        (df['name'].is_not_null()) &
        (df['nameGen'] != True) & 
        (df['description'].is_null()) & 
        (df['ryDescIsAllChine'])
    )['ryDesc'].to_list()

Since this is a null set, no further processing is required.

## Set `visitableAsOf` date

In [189]:
df = df.cast({pl.Datetime: pl.Date})
df = df.with_columns(
    visitableAsOf=pl.col('rxvisitableAsOfExtracted')
)

## Set `imageUrl` 

In [190]:
df = df.with_columns(
    imageUrl=pl.col('rxImgUrlExtracted')
)

## Review Parsing Results

In [191]:
col_order = [
'id',

'name',
'nameGen',
'name__zh-hant',
'nameGen__zh-hant',

'description',
'descriptionGen',
'description__zh-hant',
'descriptionGen__zh-hant',

'rawName',
'ryDesc',

'visitableAsOf',
'imageUrl',

'rxvisitableAsOfExtracted',
'rxImgUrlExtracted',

'rxNameIsAllEnglish',
'rxNameIsAllChinese',
'rxNameIsChineseAndEnglish',
'rxNameIsCoordinates',
'rxNameIsAddress',

'ryDescIsAllEnglish',
'ryDescIsAllChine',
'ryDescIsChineseAndEnglish',
'rxDescIsImgHtml',
'rxDescIscapturedDate',

'rawDesc',
'rawX',
'rawY',
'rawGxml'
]

df = df.select(col_order)

In [192]:
# df[
#     'name',
#     'nameGen',
#     'name__zh-hant',
#     'nameGen__zh-hant',
#     'description',
#     'descriptionGen',
#     'description__zh-hant',
#     'descriptionGen__zh-hant',
#     'rawName',
#     'ryDesc',
#     'rxNameIsAllChinese',
#     'ryDescIsAllChine'
# ].filter(
#     # pl.col('name').is_null()
#     True
# )

# Geocoding

## Fill missing Latitude & Longitude coordinates

Because the Dataframe is sorted by Latitude, the rows with missing values show up first with a `0` value. There are five of them.

In [193]:
# Addresses for GhostSigns with missing coordinates
df['rawName'].to_list()[:5]

['31 Nga Tsin Long Rd',
 '67 Kai Tak Rd',
 '42 Kai Tak Rd',
 '34 Kai Tak Rd',
 'Sung Wong Toi Garden']

In [194]:
original = df.filter(df['rawX'] == 0)['rawName'].to_list()
encodings = [
    (22.328725033185375, 114.18915132846354),
    (22.330166285778667, 114.19210082344293),
    (22.32950210461055, 114.19222392266883),
    (22.32935639108943, 114.1922340848458),
    (22.324942897750564, 114.18932768694137)
]

lats = [lat for (lat,lng) in encodings]
lngs = [lng for (lat,lng) in encodings]

address_to_lat = dict(zip(original,lats))
address_to_lng = dict(zip(original,lngs))

df = df.with_columns(
    pl.when(
        df['rawX'] == 0
    ).then(
        pl.col('rawName').replace_strict(address_to_lng, default=None)
    ).otherwise(
        pl.col('rawX')
    ).alias('longitude'),
    pl.when(
        df['rawY'] == 0
    ).then(
        pl.col('rawName').replace_strict(address_to_lat, default=None)
    ).otherwise(
        pl.col('rawY')
    ).alias('latitude'), 
)

In [195]:
# Result, with unprocessed rows
# df.select('name','longitude','latitude').head(10)   

## Reverse Geocoding - Get Address from Coordinates

### Setup Google Maps client with our API Key

In [196]:
env_vars = {}
with open('../.env') as f:
    for line in f:
        if line.startswith('#') or not line.strip():
            continue
        key, value = line.strip().split('=', 1)
        env_vars[key] = value

gmaps = googlemaps.Client(env_vars['PUBLIC_GOOGLEMAPS_KEY'])

### Define how the Google Maps API response will be used in our data schema

In [197]:
geoDB = {}

# Mapping from Google API address component name, to our component names
component_prefix = {
    'administrative_area_level_1' : 'administrativeAreaLevel1',
    'country' : "country",
    'neighborhood': "neighbourhood",
    'premise' : "premise",
    'route' : 'route',
    'street_number': 'streetNumber',
    'plus_code' : "plusCode",
    'subpremise' : 'subPremise',
    'intersection' : 'intersection'
}

# Template for Address properties (Formatted Address and Components in EN, ZH-HANT and ZH-HANS, and Metadata)
AddressProperties = {
  "distanceFromPoint" : None,
  'displayAddress' : None,
  'displayAddress__zh-hant' : None,
  'displayAddress__zh-hans' : None,
  'displayAddressGen' : True,
  'displayAddressGen__zh-hant' : True,
  'displayAddressGen__zh-hans' : True,
  "formattedAddress" : None,
  "formattedAddress__zh-hant" : None,
  "formattedAddress__zh-hans" : None,
  "plusCode" : None,
  "plusCode__zh-hant" : None,
  "plusCode__zh-hans" : None,
  "subPremise" : None,
  "subPremise__zh-hant" : None,
  "subPremise__zh-hans" : None,
  "premise" : None,
  "premise__zh-hant" : None,
  "premise__zh-hans" : None,
  "streetNumber" : None,
  "streetNumber__zh-hant" : None,
  "streetNumber__zh-hans" : None,
  "route" : None,
  "route__zh-hant" : None,
  "route__zh-hans" : None,
  "neighbourhood" : None,
  "neighbourhood__zh-hant" : None,
  "neighbourhood__zh-hans" : None,
  "administrativeAreaLevel1" : None,
  "administrativeAreaLevel1__zh-hant" : None,
  "administrativeAreaLevel1__zh-hans" : None,
  "country" : None,
  "country__zh-hant" : None,
  "country__zh-hans" : None,
  "googlePlaceId" : None,
  "geocoder" : None,
  "reverseGen" : False,
  "forwardGen" : False,
} 

def parse_to_address_properties(id, result, src_latlng:tuple, lang="en", geocoder='googlemaps', reverse_encode=True):
    lang_suffix = {
        "en": "",
        "zh-hk": "__zh-hant",
        "zh-cn": "__zh-hans",
    }.get(lang.lower())


    address_properties = AddressProperties.copy() if id not in geoDB else geoDB[id]

    if not len(result) > 0:
        raise IndexError
    
    result = result[0]
    
    for comp in result['address_components']:
        for t in comp['types']:
            if t in ['political','transit_station','bus_station','tourist_attraction', 'point_of_interest', 'establishment', 'park']:
                continue
            field = f'{component_prefix[t]}{lang_suffix}'
            address_properties[field] = comp['long_name']
            
    field = f'formattedAddress{lang_suffix}'
    address_properties[field] = result["formatted_address"]

    field = f'displayAddress{lang_suffix}'
    address_properties[field] = result["formatted_address"]
    
    field = 'distanceFromPoint'
    latlng = result["geometry"]["location"]
    address_properties[field] = int(hs.haversine(
        src_latlng,
        (latlng['lat'], latlng['lng']),
        unit=Unit.METERS
    ))
    
    field = "googlePlaceId"
    address_properties[field] = result["place_id"]

    field = "geocoder"
    address_properties[field] = geocoder

    field = "reverseGen"
    address_properties[field] = reverse_encode

    field = "forwardGen"
    address_properties[field] = reverse_encode == False

    address_properties['id'] = id
    
    geoDB[id] = address_properties

### Request Google Maps API for each LAT,LNG coordinate, and parse the result to Address Properties

We cache the API response so we can use that on subsequent runs

In [198]:
for row in df.select('id','rawName','latitude','longitude').rows(named=True):
    print(f"GEOFEATURE :: {row['id']} :: {row['rawName']}")
    
    # Look up an address with reverse geocoding
    for lang in ['en','zh-HK','zh-CN']:
        JSONPATH = f'data/{row['id']}-{lang}-gmaps-reverse-geocoded.json'
        if not os.path.isfile(JSONPATH):
            reverse_geocode_result = gmaps.reverse_geocode(
                (row['latitude'], row['longitude']),
                result_type=[
                    'subpremise',
                    'premise',
                    'establishment',
                    'street_number',
                    'street_address',
                    'point_of_interest',
                    'parking',
                    'colloquial_area',
                    'sublocality',
                    'plus_code'

                ],
                location_type=['ROOFTOP','RANGE_INTERPOLATED','GEOMETRIC_CENTER'],
                language=lang
            )

            with open(JSONPATH, 'w', encoding='utf-8') as f:
                json.dump(reverse_geocode_result, f, ensure_ascii=False, indent=4)
        else:
            # Open and read the JSON file
            with open(JSONPATH, 'r') as file:
                reverse_geocode_result = json.load(file)
        
        parse_to_address_properties(row['id'], reverse_geocode_result, (row['latitude'], row['longitude']), lang=lang)

GEOFEATURE :: oRF5wDrHYk6Uoa2JeUjQg2 :: 31 Nga Tsin Long Rd
GEOFEATURE :: HS392WkyEvD8HM2Xb8FMUQ :: 67 Kai Tak Rd
GEOFEATURE :: fGM9VAHMzm4MreR4YoDVRX :: 42 Kai Tak Rd
GEOFEATURE :: GkBYLjuBgu7CGQYBcMogXs :: 34 Kai Tak Rd
GEOFEATURE :: DvAff69iG3Jhn84gwVobiA :: Sung Wong Toi Garden
GEOFEATURE :: 2u9Dq3woi5Kco3byib93mZ :: Shan Tong Vegetable Station
GEOFEATURE :: QiV7p6KQTwZxscrdxfV9eX :: 榮發行
GEOFEATURE :: Udih7uaLLYsjEr5HVJu3Ga :: 元祥樓
GEOFEATURE :: WJAuYEZcwcvmqHzPtLDpZE :: 有限公司 under newer 福音護理院 sign
GEOFEATURE :: igpwZHtigwAuaFbMdBsmA5 :: Sam Kong Chinese Products Palimpsest 
GEOFEATURE :: Z5MDPMP5wXeFFjZYddbFGU :: Illegible blue sign on tile 中心
GEOFEATURE :: 8FktfduLzg8TfFWVqbpLnN :: Illegible church or union sign
GEOFEATURE :: nvTWmeTDAtDN8Jjnwgbptg :: 水華實業有限公司
GEOFEATURE :: mCsNPSt8ppEwAZL8u4xhJA :: Illegible white and black sign
GEOFEATURE :: X6PaudpDaDXShsmVMbH6JG :: Red and yellow furniture sign
GEOFEATURE :: 8wTe5j6hzPJdFPsk3cK9Tv :: Man Fat Co. Old Sign and illegible sign on 

In [199]:
# Example result
# geoDB['Czi8oDy7NuZwAtrLhPFkXP']

### Merge all Address Properties with Dataframe

In [200]:
df_geocoded = pl.from_dicts(list(geoDB.values()))
df = df.join(df_geocoded, on="id")

In [201]:
# Example result
# df.filter(pl.col('id')=='Czi8oDy7NuZwAtrLhPFkXP')

## Forward Geocoding - Get Address Components from Address

The idea is that the address that was originally captured in the dataset is likely to be more accurate than the reverse geocoded address obtained from the coordinates, so we will overwrite the Address Properties of those records where an address was provided.

### Request Google Maps API for each 'rawName' that was an address, and parse the result to Address Properties

We cache the API response so we can use that on subsequent runs

In [202]:
geoDB = {}

for row in df.select('id','rawName','rxNameIsAddress', 'latitude', 'longitude').rows(named=True):
    
    # Only encode raw names which were also addresses
    if not row['rxNameIsAddress']:
        continue
        
    print(f"GEOFEATURE :: {row['id']} :: {row['rawName']}")
    
    # Look up an address with forward geocoding
    for lang in ['en', 'zh-HK', 'zh-CN']:
        JSONPATH = f'data/{row['id']}-{lang}-gmaps-forward-geocoded.json'
        
        if not os.path.isfile(JSONPATH):
            print('> Google API?')
            geocode_result = gmaps.geocode(
                row['rawName'],
                components={"country":"HK"},
                language=lang
            )

            with open(JSONPATH, 'w', encoding='utf-8') as f:
                json.dump(geocode_result, f, ensure_ascii=False, indent=4)
        else:
            # Open and read the JSON file
            with open(JSONPATH, 'r') as file:
                geocode_result = json.load(file)
                
        parse_to_address_properties(row['id'], geocode_result, (row['latitude'], row['longitude']), lang=lang, reverse_encode=False)

GEOFEATURE :: oRF5wDrHYk6Uoa2JeUjQg2 :: 31 Nga Tsin Long Rd
GEOFEATURE :: HS392WkyEvD8HM2Xb8FMUQ :: 67 Kai Tak Rd
GEOFEATURE :: fGM9VAHMzm4MreR4YoDVRX :: 42 Kai Tak Rd
GEOFEATURE :: GkBYLjuBgu7CGQYBcMogXs :: 34 Kai Tak Rd
GEOFEATURE :: a3vcMhgk26WLU8qVPpFbHf :: 86 Castle Peak Rd - Yuen Long
GEOFEATURE :: CPE49czkq3ExN8AzWuP3qB :: 79 Yeung Uk Rd
GEOFEATURE :: B4zefs24vB4dwMqtETg9Uf :: 58 Catchick St
GEOFEATURE :: g75tway7j5FHkAcRMWb9M9 :: 23 Catchick St
GEOFEATURE :: m7NhfixXRzNyPtdZfscfHu :: 328 Queen's Rd W
GEOFEATURE :: dQAb9HheBKb456PhbQXwTN :: 52 Second St
GEOFEATURE :: b5xgzdeidUsg7edwKBCBx9 :: 330 Queen's Rd W
GEOFEATURE :: YsZZHso25jpZZiu7Vaqwtg :: 23 Sutherland St
GEOFEATURE :: Xtf28aBiQf3WjaNRAKUMwK :: 86 Shek Pai Wan Road
GEOFEATURE :: LaGV2gE4KvwbE2s9nV7dpC :: 89 Shek Pai Wan Road
GEOFEATURE :: bjDWVmrswSBcXtWBxxgfMx :: 10 Tin Wan Street
GEOFEATURE :: fzMrED3D4axY4XeMnwqhpm :: 97 Shek Pai Wan Road
GEOFEATURE :: EKbpgZVzMQy5o5zsfSgggC :: 11 Bonham Strand W
GEOFEATURE :: YSJAg

### Define the merge logic for how the new Address Properties should overwrite the columns in the Dataframe

In [203]:
# The new address components have an intersection component, so we need to backfill this into our original Dataframe
df = df.with_columns(
    pl.lit(None).alias('intersection'),
    pl.lit(None).alias('intersection__zh-hant'),
    pl.lit(None).alias('intersection__zh-hans'),
)

In [204]:
def update(self:pl.DataFrame, df_other:pl.DataFrame,  join_columns:list[str]) -> pl.DataFrame:
    '''Updates dataframe with the values in df_other after joining on the join_columns'''
    
    # The columns that will be updated
    columns = [c for c in df_other.columns if c not in join_columns]
    
    df_merged = (self
        .join(df_other, how='left', on=join_columns, suffix='_NEW')      
        .with_columns(**{
            c: pl.coalesce([pl.col(c+'_NEW'), pl.col(c)]) for c in columns # <-This updates the columns
        }).select(
            pl.all().exclude('^.*_NEW$') # <- this drops the temporary '*_NEW' columns
           )
       )    
    return df_merged

### Merge the Forward Encoded Address Properties with Dataframe

In [205]:
df_forward_encoded = pl.from_dicts(list(geoDB.values()))
# df.join(df_forward_encoded, on="id", how='left', coalesce=True)
df = update(df, df_forward_encoded, join_columns=['id'])

# Exporting

## Review Result

In [206]:
col_order = [
'id',

'name',
'name__zh-hant',
'nameGen',
'nameGen__zh-hant',
    
'description',
'description__zh-hant',
'descriptionGen',
'descriptionGen__zh-hant',

'rawName',
'ryDesc',

'longitude',
'latitude',
'distanceFromPoint',

'displayAddress',
'displayAddress__zh-hant',
'displayAddress__zh-hans',
'displayAddressGen',
'displayAddressGen__zh-hant',
'displayAddressGen__zh-hans',
    
'formattedAddress',
'formattedAddress__zh-hant',
'formattedAddress__zh-hans',

'visitableAsOf',
'imageUrl',

'plusCode',
'plusCode__zh-hant',
'plusCode__zh-hans',
'subPremise',
'subPremise__zh-hant',
'subPremise__zh-hans',
'premise',
'premise__zh-hant',
'premise__zh-hans',
'streetNumber',
'streetNumber__zh-hant',
'streetNumber__zh-hans',
'intersection',
'intersection__zh-hant',
'intersection__zh-hans',
'route',
'route__zh-hant',
'route__zh-hans',
'neighbourhood',
'neighbourhood__zh-hant',
'neighbourhood__zh-hans',
'administrativeAreaLevel1',
'administrativeAreaLevel1__zh-hant',
'administrativeAreaLevel1__zh-hans',
'country',
'country__zh-hant',
'country__zh-hans',

'googlePlaceId',
'geocoder',
'reverseGen',
'forwardGen',

'rxvisitableAsOfExtracted',
'rxImgUrlExtracted',

'rxNameIsAllEnglish',
'rxNameIsAllChinese',
'rxNameIsChineseAndEnglish',
'rxNameIsCoordinates',
'rxNameIsAddress',

'ryDescIsAllEnglish',
'ryDescIsAllChine',
'ryDescIsChineseAndEnglish',
'rxDescIsImgHtml',
'rxDescIscapturedDate',

'rawDesc',
'rawX',
'rawY',
'rawGxml'
]

df = df.select(col_order)

In [207]:
cfg.set_tbl_rows(70)

df.row(44, named=True)

{'id': '6gNePfh82C5hbHwK765qTR',
 'name': 'Shanghainese Barber',
 'name__zh-hant': None,
 'nameGen': False,
 'nameGen__zh-hant': None,
 'description': None,
 'description__zh-hant': None,
 'descriptionGen': None,
 'descriptionGen__zh-hant': None,
 'rawName': 'Shanghainese Barber',
 'ryDesc': None,
 'longitude': 114.1367873,
 'latitude': 22.2530287,
 'distanceFromPoint': 14,
 'displayAddress': 'Wah Fu (II) Estate Wah King House, 10號 Wah Cheung St, Waterfall Bay, Hong Kong',
 'displayAddress__zh-hant': '香港瀑布灣華昌街10號華富(二)邨華景樓',
 'displayAddress__zh-hans': '香港瀑布灣華昌街10號華富(二)邨華景樓',
 'displayAddressGen': True,
 'displayAddressGen__zh-hant': True,
 'displayAddressGen__zh-hans': True,
 'formattedAddress': 'Wah Fu (II) Estate Wah King House, 10號 Wah Cheung St, Waterfall Bay, Hong Kong',
 'formattedAddress__zh-hant': '香港瀑布灣華昌街10號華富(二)邨華景樓',
 'formattedAddress__zh-hans': '香港瀑布灣華昌街10號華富(二)邨華景樓',
 'visitableAsOf': datetime.date(2023, 10, 1),
 'imageUrl': 'https://mymaps.usercontent.google.com/hostedi

In [208]:
# cfg.set_tbl_rows(10)
# df.select('name','name__zh-hant','longitude','latitude', 'formattedAddress','formattedAddress__zh-hant','formattedAddress__zh-hans')

## Save Result

In [209]:
df.write_csv(DATADEST)

### Provisional Data for App

In [210]:
# df_temp.columns

In [211]:
df_temp = df

### Fill in missing English names

In [212]:
df_temp = df_temp.with_columns(
    pl.when(
        pl.col('name').is_null()    
    ).then(
        pl.col('rawName')
    ).otherwise(
        pl.col('name')
    ).alias(
        'name'
    )
)

## Fill `graphemes` with Traditional Chinese title

In [213]:
df_temp = df_temp.with_columns(
    pl.when(
        pl.col('name__zh-hant').is_not_null()    
    ).then(
        pl.col('name__zh-hant')
    ).otherwise(
        None
    ).alias(
        'graphemes'
    )
)

In [214]:
selected_cols = [not (x.startswith('raw') | x.startswith('rx') | x.startswith('ry')) for x in df_temp.columns]

In [215]:
df_temp[selected_cols].write_json(JSONDEST)

In [219]:
# for k in df_temp[selected_cols].to_dicts()[0].keys():
#     print(k)

In [218]:
tijptjikId = "f16249f5-c045-496a-a441-16ad07600ce2"
geoFeatures = []
for r in df_temp[selected_cols].to_dicts():
    geoFeatures.append(
        {
            "id" : r['id'],
            "geometry" : {
                "type": "Point",
                "coordinates": [r['longitude'], r['latitude']]
            },
            "properties" : {
                # Title
                'title': r['name'],
                'title__zh-hant': r['name__zh-hant'],
                'title__zh-hans': r['name__zh-hant'],
                'titleGen': r['nameGen'],
                'titleGen__zh-hant': r['nameGen__zh-hant'],
                'titleGen__zh-hans': r['nameGen__zh-hant'],
                
                # Description
                'description': r['description'],
                'description__zh-hant': r['description__zh-hant'],
                'description__zh-hans': r['description__zh-hant'],
                'descriptionGen': r['descriptionGen'],
                'descriptionGen__zh-hant' : r['descriptionGen__zh-hant'],
                'descriptionGen__zh-hans' : r['descriptionGen__zh-hant'],
                
                # Misc
                'grade' : choice([1,2,3,4,5]),
                
                # Custom
                "graphemes" : r['graphemes'],
                "size" : choice(['small','medium','large']),
                "material" : choice(["painted on cement", "painted on metal", "painted on brick", "painted on tile", "painted over", "acrylic", "metal", "wood", "terrazzo", "stone", "molded cement", "zinc stenciled"]),
                "visibility" : choice(['revenant', 'physical', 'palimpsest', 'uncovering']),
            },
            "addressProperties":{
                "distanceFromPoint" : r["distanceFromPoint"],
                "formattedAddress" : r["formattedAddress"],
                "formattedAddress__zh-hant" : r["formattedAddress__zh-hant"],
                "formattedAddress__zh-hans" : r["formattedAddress__zh-hant"],
                "plusCode" : r["plusCode"],
                "plusCode__zh-hant" : r["plusCode__zh-hant"],
                "plusCode__zh-hans" : r["plusCode__zh-hans"],
                "subPremise" : r["subPremise"],
                "subPremise__zh-hant" : r["subPremise__zh-hant"],
                "subPremise__zh-hans" : r["subPremise__zh-hans"],
                "premise" : r["premise"],
                "premise__zh-hant" : r["premise__zh-hant"],
                "premise__zh-hans" : r["premise__zh-hans"],
                "streetNumber" : r["streetNumber"],
                "streetNumber__zh-hant" : r["streetNumber__zh-hant"],
                "streetNumber__zh-hans" : r["streetNumber__zh-hans"],
                "route" : r["route"],
                "route__zh-hant" : r["route__zh-hant"],
                "route__zh-hans" : r["route__zh-hans"],
                "intersection" : r["intersection"],
                "intersection__zh-hant" : r["intersection__zh-hant"],
                "intersection__zh-hans" : r["intersection__zh-hans"],
                "neighbourhood" : r["neighbourhood"],
                "neighbourhood__zh-hant" : r["neighbourhood__zh-hant"],
                "neighbourhood__zh-hans" : r["neighbourhood__zh-hans"],
                "administrativeAreaLevel1" : r["administrativeAreaLevel1"],
                "administrativeAreaLevel1__zh-hant" : r["administrativeAreaLevel1__zh-hant"],
                "administrativeAreaLevel1__zh-hans" : r["administrativeAreaLevel1__zh-hans"],
                "country" : r["country"],
                "country__zh-hant" : r["country__zh-hant"],
                "country__zh-hans" : r["country__zh-hans"],
                "googlePlaceId" : r["googlePlaceId"],
                "addressGeocoder": r["geocoder"],
                "addressReverseGen": r["reverseGen"],
                "addressForwardGen": r["forwardGen"],
            },
            "geoCollectionId" : "85d503ae-9ea6-43d7-a40c-0853a722313d",
            "contributorId" : tijptjikId,
            "publisherId" : tijptjikId,
            "isPublished" : True,
            "isHistoric" : False,
            "isVisitable" : True,
            "visitableAsOf" : r['visitableAsOf']
        }
    )
    

In [239]:
def default(o):
    if type(o) is date:
        return o.isoformat()

def jsondump(o, f):
    return json.dump(o, f, ensure_ascii=False, indent=4, default=default)

with open(JSONDEST, 'w', encoding='utf-8') as f:
    jsondump(geoFeatures, f)

# Google Translate Cantonese Experiment

In [ ]:
# Initialize Translation client
def translate(
    contents: str | list[str],
    source_lang : str = "en",
    target_lang : str = "zh-HK",
    project_id: str = "ghostmapper",
    model_id : str = 'NM3b0572414904af9c',
    location : str = 'us-central1',
    print_results: bool = False
) -> google_translate.TranslationServiceClient:
    """Translating text with Google Translation API."""

    client = google_translate.TranslationServiceClient()

    parent = f"projects/{project_id}/locations/{location}"
    model = f"{parent}/models/{model_id}"

    if not type(contents) == list:
        contents = [contents]
    
    # Translate text from English to Hong Kong Chinese with a custom model
    # See https://console.cloud.google.com/translation/models/list?project=ghostmapper 
    response = client.translate_text(
        request={
            "contents": contents,
            "parent": parent,
            "model": model,
            "source_language_code": source_lang,
            "target_language_code": target_lang,
            "mime_type": "text/plain",  # mime types: text/plain, text/html
        }
    )

    # Display the translation for each input text provided
    if print_results:
        print(f'TRANSLATION >>> {source_lang} > {target_lang}')
        for translation in response.translations:
            print(f'{text} --> {translation.translated_text}')

    return [t.translated_text for t in response.translations]

translate_en_to_zhhk = lambda text, print_results: translate(text, "en", "zh-HK", print_results=print_results)
# translate_en_to_zhcn = lambda text, print_results: translate(text, "en", "zh-CN")
# translate_zhhk_to_en = lambda text, print_results: translate(text, "zh-HK", "en")
# translate_zhcn_to_en = lambda text, print_results: translate(text, "zh-CN", "en")
# translate_zhhk_to_zhcn = lambda text, print_results: translate(text, "zh-HK", "zh-CN")


text = 'Hello World, let\'s guide you through the wonderful world of Hong Kong'

translate_en_to_zhhk(text, True)


## Temporary Interpolated Results

In [ ]:
# DESC : Central Pier 5
# DESC : DHL Express Service Point
# DESC : New Treasure Centre

In [ ]:
# "'Strong Hardware'",
#  "'Remove'",
#  "'Songji Daxin Fashion'",
 
#  "'WenJi decoration project' in yellow",
#  "'Ming Hua Noodle House'",
#  "'Hongchang Tea House'",
#  "'He Kee Hardware'",
#  "'Shengli'",
#  "'Xinglu Shiduo'",
#  "'Pan Ji Wood Paint'",
#  "'Telemo Pump'",
#  "'Fourth floor of 'Hanglebie'",
#  "'Oshidahexie'",
#  "'Ya Nguyen'",
#  "'Huang Ti Nong's Western Clothes'",